# 10. Data nyata dan fine-tuning BERT kecil

Modul tambahan ini menghubungkan alur PyTorch sebelumnya dengan tokenizer subword dan model pralatih dari Hugging Face. Kita menggunakan SST-2 berbahasa Inggris agar selaras dengan model `prajjwal1/bert-tiny`.

**Prasyarat:** modul 01, 04, dan 06. Instal `requirements-hf.txt` melalui terminal sebelum menjalankan notebook. Unduhan dataset dan bobot memerlukan internet. CPU dapat digunakan untuk subset kecil.

**Tujuan:** memisahkan data dengan benar, memakai tokenizer pasangan checkpoint, melakukan fine-tuning menggunakan loop PyTorch yang eksplisit, dan memeriksa penyimpanan model.

Ini eksperimen subset pembelajaran. Hasilnya tidak dapat dibandingkan langsung dengan skor benchmark SST-2.

In [1]:
from pathlib import Path
import sys
# Lokal: buka dari root repo, folder nlp, atau nlp/notebooks.
# Colab: ambil paket kursus jika belum tersedia.
candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for base in candidates for p in (base, base / "nlp")
             if (p / "nlp_course").is_dir()), None)
if ROOT is None and "google.colab" in sys.modules:
    import subprocess
    target = Path("/content/pytorch-deep-learning-nlp")
    if not target.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                        "--sparse", "--branch", "nlp-learning-path",
                        "https://github.com/FeliksMakarios/pytorch-deep-learning.git",
                        str(target)], check=True)
        subprocess.run(["git", "sparse-checkout", "set", "nlp"], cwd=target, check=True)
    ROOT = target / "nlp"
if ROOT is None or not (ROOT / "nlp_course").is_dir():
    raise RuntimeError("Folder nlp_course tidak ditemukan. Ikuti petunjuk README nlp.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import torch
from torch import nn
from nlp_course.data import tokenize, build_vocab, encode, read_rows, loaders, collate_batch
from nlp_course.models import MeanClassifier, RecurrentClassifier, TinyTransformer
from nlp_course.engine import seed_all, fit, run_epoch, metrics, save_mean, load_mean, predict
seed_all(42)
torch.set_num_threads(1)
device = "cuda" if torch.cuda.is_available() else "cpu"
ARTIFACTS = ROOT / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)
print("PyTorch:", torch.__version__, "Perangkat:", device)

PyTorch: 2.14.0+cu130 Perangkat: cpu


## 1. Memuat dependensi dan konfigurasi

BERT kecil mengurangi biaya demonstrasi. Kepala klasifikasi dibuat untuk tugas dua kelas dan memerlukan pelatihan. Pilihan ukuran subset dan epoch di bawah bertujuan menjaga eksperimen tetap ringan.

In [2]:
import os
# Gunakan checkpoint sumber tanpa memulai layanan konversi eksternal.
os.environ["DISABLE_SAFETENSORS_CONVERSION"] = "1"
import copy, json
from datasets import load_dataset
from transformers import (AutoTokenizer, BertTokenizer, BertConfig, BertForSequenceClassification,
                          AutoModelForSequenceClassification, DataCollatorWithPadding)
from huggingface_hub import hf_hub_download
from torch.utils.data import Dataset, DataLoader
MODEL_ID = "prajjwal1/bert-tiny"
N_TRAIN = 512
N_VAL = 128
N_HOLDOUT = 128
EPOCHS = 2
MAX_LENGTH = 128
seed_all(42)
print("Model:", MODEL_ID)

Model: prajjwal1/bert-tiny


## 2. Dataset dan protokol evaluasi

SST-2 menyediakan kalimat dan label sentimen. Kita mengambil data latih dari split train. Split validation resmi dibagi menjadi validasi lokal dan data tahanan lokal untuk demonstrasi. Data tahanan ini bukan test resmi benchmark.

SST-2 berisi cuplikan kalimat yang dapat berkaitan. Pemeriksaan duplikat persis membantu, tetapi tidak membuktikan kemandirian semantik seluruh contoh. Jangan menganggap evaluasi subset ini sebagai replikasi benchmark.

In [3]:
raw = load_dataset("stanfordnlp/sst2")
train_pool = raw["train"].shuffle(seed=42)
held_pool = raw["validation"].shuffle(seed=42)
def clean_key(text):
    return " ".join(text.lower().split())
# Sisihkan kandidat tahanan dahulu, kemudian cegah duplikat persis ke latih.
held_rows = []
seen = set()
for row in held_pool:
    key = clean_key(row["sentence"])
    if key and key not in seen:
        held_rows.append(dict(row))
        seen.add(key)
    if len(held_rows) == N_VAL + N_HOLDOUT:
        break
val_rows = held_rows[:N_VAL]
holdout_rows = held_rows[N_VAL:]
train_rows = []
for row in train_pool:
    key = clean_key(row["sentence"])
    if key and key not in seen:
        train_rows.append(dict(row))
        seen.add(key)
    if len(train_rows) == N_TRAIN:
        break
assert len(train_rows)==N_TRAIN and len(holdout_rows)==N_HOLDOUT
print("Latih/validasi/tahanan lokal:",len(train_rows),len(val_rows),len(holdout_rows))
print(train_rows[0])

Latih/validasi/tahanan lokal: 512 128 128
{'idx': 32326, 'sentence': 'klein , charming in comedies like american pie and dead-on in election , ', 'label': 1}


## 3. Tokenizer dari checkpoint yang sama

Tokenizer menghasilkan input_ids dan attention_mask. Pada tokenizer Hugging Face, attention_mask bernilai 1 untuk token nyata dan 0 untuk padding. Token khusus ditangani tokenizer. Jangan membentuk vocabulary sendiri untuk bobot BERT yang sudah ada.

Checkpoint BERT-Tiny ini menyimpan vocabulary lama dalam vocab.txt. Pada Transformers 5, kita memuat berkas tersebut secara eksplisit dengan BertTokenizer. Indeks vocabulary tetap berasal dari checkpoint sumber.

In [4]:
# Checkpoint lama menyediakan vocab.txt. Transformers 5 memakai argumen vocab.
vocab_path = hf_hub_download(MODEL_ID, "vocab.txt")
tokenizer = BertTokenizer(vocab=vocab_path, do_lower_case=True)
example = tokenizer("This movie is surprisingly good.")
print(example)
print(tokenizer.convert_ids_to_tokens(example["input_ids"]))
class EncodedText(Dataset):
    def __init__(self, rows):
        self.rows = rows
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, index):
        row = self.rows[index]
        item = tokenizer(row["sentence"],truncation=True,max_length=MAX_LENGTH)
        item["labels"] = row["label"]
        return item
collator = DataCollatorWithPadding(tokenizer=tokenizer)
train_loader = DataLoader(EncodedText(train_rows),batch_size=16,shuffle=True,
                          collate_fn=collator,generator=torch.Generator().manual_seed(42))
val_loader = DataLoader(EncodedText(val_rows),batch_size=32,collate_fn=collator)
holdout_loader = DataLoader(EncodedText(holdout_rows),batch_size=32,collate_fn=collator)
batch = next(iter(train_loader))
print({key:value.shape for key,value in batch.items()})

{'input_ids': [101, 2023, 3185, 2003, 10889, 2204, 1012, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1]}
['[CLS]', 'this', 'movie', 'is', 'surprisingly', 'good', '.', '[SEP]']
{'input_ids': torch.Size([16, 25]), 'token_type_ids': torch.Size([16, 25]), 'attention_mask': torch.Size([16, 25]), 'labels': torch.Size([16])}


## 4. Memuat model dan menetapkan optimizer

Semua bobot dapat diperbarui pada demonstrasi fine-tuning ini. Untuk percobaan frozen encoder, bekukan `model.base_model.parameters()` sebelum membuat optimizer. Model dan batch dipindahkan ke perangkat yang sama.

Konfigurasi lama BERT-Tiny belum memuat model_type. Kita menyatakan BertConfig dan BertForSequenceClassification secara eksplisit. Setelah save_pretrained, konfigurasi baru memuat jenis model sehingga AutoModel dapat digunakan saat pemuatan ulang.

In [5]:
# Config checkpoint lama belum memiliki model_type. Nyatakan kelas BERT secara eksplisit.
config = BertConfig.from_pretrained(MODEL_ID, num_labels=2)
model = BertForSequenceClassification.from_pretrained(MODEL_ID, config=config, use_safetensors=False)
model.to(device)
optimizer = torch.optim.AdamW(model.parameters(),lr=2e-5,weight_decay=0.01)
print("Parameter:",sum(p.numel() for p in model.parameters()))

Loading weights: 100%|##########| 39/39 [00:00<00:00, 13089.37it/s]
Parameter: 4386178


## 5. Loop PyTorch dan pemilihan checkpoint

Model menghitung cross entropy jika labels diberikan. Eval menonaktifkan dropout. torch.set_grad_enabled menentukan pencatatan gradien. Checkpoint dipilih hanya berdasarkan loss validasi lokal.

In [6]:
def bert_epoch(loader, training=False):
    model.train(training)
    total_loss,n,truth,predicted = 0.,0,[],[]
    with torch.set_grad_enabled(training):
        for batch in loader:
            batch = {key:value.to(device) for key,value in batch.items()}
            output = model(**batch)
            if training:
                optimizer.zero_grad()
                output.loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(),1.)
                optimizer.step()
            size = batch["labels"].shape[0]
            total_loss += output.loss.item()*size
            n += size
            truth.extend(batch["labels"].cpu().tolist())
            predicted.extend(output.logits.argmax(-1).detach().cpu().tolist())
    return {"loss":total_loss/n,**metrics(truth,predicted)}
best_loss,best_state = float("inf"),None
history=[]
for epoch in range(EPOCHS):
    train_result=bert_epoch(train_loader,training=True)
    val_result=bert_epoch(val_loader)
    history.append({"epoch":epoch+1,"train":train_result,"val":val_result})
    print(history[-1])
    if val_result["loss"] < best_loss:
        best_loss=val_result["loss"]
        best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
model.load_state_dict(best_state)
print("Tahanan lokal:",bert_epoch(holdout_loader))

{'epoch': 1, 'train': {'loss': 0.6865152325481176, 'accuracy': 0.546875, 'macro_f1': 0.47798267006874084, 'confusion': [[47, 168], [64, 233]]}, 'val': {'loss': 0.6822601854801178, 'accuracy': 0.5625, 'macro_f1': 0.4589371979236603, 'confusion': [[8, 51], [5, 64]]}}
{'epoch': 2, 'train': {'loss': 0.6761536281555891, 'accuracy': 0.56640625, 'macro_f1': 0.4660152196884155, 'confusion': [[34, 181], [41, 256]]}, 'val': {'loss': 0.6801094859838486, 'accuracy': 0.578125, 'macro_f1': 0.5215947031974792, 'confusion': [[15, 44], [10, 59]]}}
Tahanan lokal: {'loss': 0.676437184214592, 'accuracy': 0.578125, 'macro_f1': 0.5272229313850403, 'confusion': [[16, 45], [9, 58]]}


## 6. Simpan model dan tokenizer bersama

save_pretrained menyimpan konfigurasi serta bobot model. Tokenizer juga harus disimpan ke folder yang sama. Pemuatan ulang diperiksa menggunakan logits pada teks yang sama.

In [7]:
destination=ARTIFACTS/"bert_sentiment"
model.save_pretrained(destination)
tokenizer.save_pretrained(destination)
restored=AutoModelForSequenceClassification.from_pretrained(destination).to(device).eval()
restored_tokenizer=AutoTokenizer.from_pretrained(destination)
model.eval()
text="This movie is enjoyable."
a=tokenizer(text,return_tensors="pt",truncation=True,max_length=MAX_LENGTH).to(device)
b=restored_tokenizer(text,return_tensors="pt",truncation=True,max_length=MAX_LENGTH).to(device)
with torch.inference_mode():
    original_scores=model(**a).logits
    restored_scores=restored(**b).logits
assert torch.allclose(original_scores,restored_scores,atol=1e-5)
print(dict(zip(["negative","positive"],restored_scores.softmax(-1)[0].cpu().tolist())))
(destination/"experiment.json").write_text(json.dumps({"model_id":MODEL_ID,"seed":42,
    "n_train":N_TRAIN,"n_val":N_VAL,"n_holdout":N_HOLDOUT,"max_length":MAX_LENGTH,
    "epochs":EPOCHS,"history":history},indent=2),encoding="utf-8")

Loading weights: 100%|##########| 41/41 [00:00<00:00, 5017.40it/s]
{'negative': 0.44652798771858215, 'positive': 0.5534720420837402}
1325


## Latihan dan pembahasan

1. Bandingkan frozen encoder dan fine-tuning penuh dengan split tetap. Model terbaik dipilih melalui validasi lokal. Tidak ada jaminan fine-tuning selalu menang pada subset kecil.
2. Tingkatkan ukuran data latih. Catat waktu dan perubahan macro-F1. Jangan mengganti subset tahanan setiap kali hasil mengecewakan.
3. Adaptasikan ke bahasa Indonesia. Pilih model dan tokenizer yang mendukung bahasa Indonesia, gunakan dataset berlisensi yang sesuai, lalu audit pembagian data dan label. BERT kecil berbahasa Inggris di sini bukan pilihan otomatis untuk bahasa Indonesia.
4. Bandingkan dengan model sederhana. Model besar perlu menunjukkan manfaat yang sepadan dengan biaya dan kebutuhan aplikasinya.

## Rujukan

- [Model card BERT-Tiny](https://huggingface.co/prajjwal1/bert-tiny)
- [Dataset card SST-2](https://huggingface.co/datasets/stanfordnlp/sst2)
- [Panduan klasifikasi teks Transformers](https://huggingface.co/docs/transformers/tasks/sequence_classification)
- [Data collator](https://huggingface.co/docs/transformers/main_classes/data_collator)

Kartu model dan dataset harus ditinjau kembali untuk ketentuan penggunaan sebelum redistribusi data atau bobot. Paket kursus menyertakan kode unduhan, bukan menyalin dataset SST-2 atau bobot BERT.